In [1]:
from __future__ import annotations
import os, time, math, random
from typing import List, Dict, Set, Tuple

import torch

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import Crippen
from rdkit.Chem import QED
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

In [2]:
DATA_PATH = "./cache_qm9/qm9_splits_vocab.pt"  

assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"

obj = torch.load(DATA_PATH)
train_sm, val_sm, test_sm = obj["splits"]
stoi, itos = obj["stoi"], obj["itos"]
cfg = obj.get("cfg", {})

print("Loaded:")
print(" Train:", len(train_sm), "Val:", len(val_sm), "Test:", len(test_sm))
print(" Vocab size:", len(itos))
print(" cfg:", cfg)

Loaded:
 Train: 120496 Val: 6694 Test: 6695
 Vocab size: 24
 cfg: {'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/qm9.csv', 'smiles_col': 'smiles', 'seed': 42, 'train_frac': 0.9, 'val_frac': 0.05, 'max_len': 80, 'cache_dir': './cache_qm9', 'cache_file': 'qm9_processed.pt'}


In [3]:
def to_mol(smiles: str):
    """Returns RDKit Mol or None."""
    return Chem.MolFromSmiles(smiles)

def canonical(smiles: str) -> str | None:
    """Canonical SMILES or None if invalid."""
    m = to_mol(smiles)
    if m is None:
        return None
    return Chem.MolToSmiles(m)

train_canon: Set[str] = set()
bad_train = 0
for s in train_sm:
    cs = canonical(s)
    if cs is None:
        bad_train += 1
    else:
        train_canon.add(cs)

print("Train canonical set size:", len(train_canon))
print("Invalid in train (should be 0):", bad_train)


Train canonical set size: 120432
Invalid in train (should be 0): 0


In [4]:
def validity(smiles_list: List[str]) -> Tuple[float, List[str]]:
    """Return validity fraction and list of valid canonical SMILES."""
    valid_canon = []
    for s in smiles_list:
        cs = canonical(s)
        if cs is not None:
            valid_canon.append(cs)
    v = len(valid_canon) / max(1, len(smiles_list))
    return v, valid_canon

def uniqueness(valid_canon: List[str]) -> float:
    """Unique fraction among valid samples."""
    if len(valid_canon) == 0:
        return 0.0
    return len(set(valid_canon)) / len(valid_canon)

def novelty(valid_canon: List[str], train_set_canon: Set[str]) -> float:
    """Novel fraction among valid samples (not in train)."""
    if len(valid_canon) == 0:
        return 0.0
    novel = sum(1 for s in valid_canon if s not in train_set_canon)
    return novel / len(valid_canon)


In [5]:
def compute_properties(valid_canon: List[str]) -> Dict[str, float]:
    """
    Computes basic properties on valid SMILES:
    - MW (molecular weight)
    - LogP (Crippen)
    - QED (drug-likeness heuristic; optional)
    Returns mean values. If none valid, returns NaNs.
    """
    if len(valid_canon) == 0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}

    mws, logps, qeds = [], [], []
    for s in valid_canon:
        m = to_mol(s)
        if m is None:
            continue
        mws.append(Descriptors.MolWt(m))
        logps.append(Crippen.MolLogP(m))
        qeds.append(QED.qed(m))

    if len(mws) == 0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}

    return {
        "mw_mean": sum(mws) / len(mws),
        "logp_mean": sum(logps) / len(logps),
        "qed_mean": sum(qeds) / len(qeds),
    }


In [6]:
def evaluate_smiles(samples: List[str], train_set_canon: Set[str]) -> Dict[str, float]:
    """
    Returns a dict of metrics:
    - n_samples
    - validity
    - uniqueness (among valid)
    - novelty (among valid)
    - mw_mean/logp_mean/qed_mean (valid only)
    """
    v, valid_canon = validity(samples)
    u = uniqueness(valid_canon)
    n = novelty(valid_canon, train_set_canon)
    props = compute_properties(valid_canon)

    out = {
        "n_samples": len(samples),
        "validity": v,
        "uniqueness": u,
        "novelty": n,
        "n_valid": len(valid_canon),
    }
    out.update(props)
    return out


In [7]:
train_subset = random.sample(train_sm, 2000)
metrics_train = evaluate_smiles(train_subset, train_canon)
metrics_train


{'n_samples': 2000,
 'validity': 1.0,
 'uniqueness': 1.0,
 'novelty': 0.0,
 'n_valid': 2000,
 'mw_mean': 122.9520145000002,
 'logp_mean': 0.3137449600000009,
 'qed_mean': 0.4670207625653176}

In [8]:
chars = [c for c in itos if c not in ("<PAD>", "<BOS>", "<EOS>")]
def random_smiles_like_string(n_chars: int) -> str:
    return "".join(random.choice(chars) for _ in range(n_chars))

random_samples = [random_smiles_like_string(random.randint(5, 30)) for _ in range(2000)]
metrics_rand = evaluate_smiles(random_samples, train_canon)
metrics_rand

{'n_samples': 2000,
 'validity': 0.0,
 'uniqueness': 0.0,
 'novelty': 0.0,
 'n_valid': 0,
 'mw_mean': nan,
 'logp_mean': nan,
 'qed_mean': nan}

In [9]:
def timed_evaluate(samples: List[str], train_set_canon: Set[str]) -> Dict[str, float]:
    t0 = time.time()
    m = evaluate_smiles(samples, train_set_canon)
    t1 = time.time()
    m["eval_seconds"] = t1 - t0
    if m["eval_seconds"] > 0:
        m["valid_per_sec_eval_only"] = m["n_valid"] / m["eval_seconds"]
    return m

timed_evaluate(train_subset, train_canon)

{'n_samples': 2000,
 'validity': 1.0,
 'uniqueness': 1.0,
 'novelty': 0.0,
 'n_valid': 2000,
 'mw_mean': 122.9520145000002,
 'logp_mean': 0.3137449600000009,
 'qed_mean': 0.4670207625653176,
 'eval_seconds': 2.218008279800415,
 'valid_per_sec_eval_only': 901.7098890992272}